In [9]:
library(dplyr)
library(randomForest)

In [10]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

We try to perform a k-fold CV with normal dataset, undersampled dataset and oversampled dataset to assess if the imbalance of target values affects the performance of RF.

In [1]:
datam<-read.csv("data_target_encoding.csv",stringsAsFactors = T)

target_variable<-match('damage_grade', colnames(datam))
nfeat <- ncol(datam)-1 #I remove 2 to remove building_id and damage_grade

nrows<-nrow(datam)

k<-10
n_trees<-20

In [25]:
set.seed(2) 
accuracy_vec <- array(0,k)


# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrows)
# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    #3.2 Take the remaining groups as a training data set
    train_data <- datam[-splits[[i]],]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    rm('model')
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.741759717585665"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.743064349027282"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.741069030351867"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.740378343118069"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.743870150800046"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.742603890871417"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.74621081309236"
  |========================================================              |  80%[1] "F1-Score Micro - 8 fo

In [31]:
set.seed(2)
accuracy_vec <- array(0,k)


# 1. Shuffle the dataset randomly and undersample
datasub_1idx <- which(datam$damage_grade == 1)
n_sub <- nrow(datasub_1)

datasub_2idx <- sample(which(datam$damage_grade == 2),n_sub)
datasub_3idx <- sample(which(datam$damage_grade == 3),n_sub)
datasub <- rbind(datam[datasub_1idx,],datam[datasub_2idx,],datam[datasub_3idx,])
datasub_idx <- sample(1:nrow(datasub))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datasub)/k)
splits <- split(datasub_idx, ceiling(seq_along(datasub_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

    #3.1 Take the group as a hold out or test data set
    test_data <- datasub[splits[[i]],]


    #3.2 Take the remaining groups as a training data set
    train_data <- datasub[-splits[[i]],]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.735738922791191"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.737065534624569"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.745555850358185"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.749801008224993"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.742106659591404"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.740912708941364"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.737463518174582"
  |========================================================              |  80%[1] "F1-Score Micro - 8 f

In [6]:
set.seed(2)

accuracy_vec <- array(0,k)
# 1. Shuffle the dataset randomly and undersample
data_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){
    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    dataoverid_1 <- which(datam[-splits[[i]],]$damage_grade == 1)
    dataoverid_2 <- which(datam[-splits[[i]],]$damage_grade == 2)
    dataoverid_3 <- which(datam[-splits[[i]],]$damage_grade == 3)
    n_over1 <- length(dataoverid_2)-length(dataoverid_1)
    n_over3 <- length(dataoverid_2)-length(dataoverid_3)

    dataover_1 <- sample(dataoverid_1,n_over1,replace=TRUE)
    dataover_3 <- sample(dataoverid_3,n_over3,replace=TRUE)
    dataover <- rbind(datam[-splits[[i]],],datam[-splits[[i]],][dataover_1,],datam[-splits[[i]],][dataover_3,])
     #3.2 Take the remaining groups as a training data set
    train_data <- dataover[,-id_variable]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |==============                                                        |  20%[1] "F1-Score Micro - 1 fold: 0.722223288118033"
  |============================                                          |  40%[1] "F1-Score Micro - 2 fold: 0.721436657009651"
  |==========================================                            |  60%[1] "F1-Score Micro - 3 fold: 0.721954682373707"
  |========================================================              |  80%[1] "F1-Score Micro - 4 fold: 0.724122714452908"
  |======================================================================| 100%[1] "F1-Score Micro - 5 fold: 0.722067655467506"
[1] "Mean F1-Score Micro: 0.722360999484361"


In [8]:
set.seed(2)

accuracy_vec <- array(0,k)
# 1. Shuffle the dataset randomly and undersample
data_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){
    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    dataoverid_1 <- which(datam[-splits[[i]],]$damage_grade == 1)
    dataoverid_2 <- which(datam[-splits[[i]],]$damage_grade == 2)
    dataoverid_3 <- which(datam[-splits[[i]],]$damage_grade == 3)
    n_over1 <- 0.75*(length(dataoverid_2)-length(dataoverid_1))
    n_over3 <- 0.25*(length(dataoverid_2)-length(dataoverid_3))

    dataover_1 <- sample(dataoverid_1,n_over1,replace=TRUE)
    dataover_3 <- sample(dataoverid_3,n_over3,replace=TRUE)
    dataover <- rbind(datam[-splits[[i]],],datam[-splits[[i]],][dataover_1,],datam[-splits[[i]],][dataover_3,])
     #3.2 Take the remaining groups as a training data set
    train_data <- dataover[,-id_variable]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |==============                                                        |  20%[1] "F1-Score Micro - 1 fold: 0.726904702519138"
  |============================                                          |  40%[1] "F1-Score Micro - 2 fold: 0.723105849849389"
  |==========================================                            |  60%[1] "F1-Score Micro - 3 fold: 0.727998311621036"
  |========================================================              |  80%[1] "F1-Score Micro - 4 fold: 0.722818057980469"
  |======================================================================| 100%[1] "F1-Score Micro - 5 fold: 0.726346489629104"
[1] "Mean F1-Score Micro: 0.725434682319827"
